# Trainer

Simple trainer for fnl prediction

In [ ]:
import os
import sys
import math
import logging
from datetime import datetime

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, Dropout, Flatten, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool

import wandb
from wandb.integration.keras import WandbMetricsLogger

from mlpng import Core
from mlpng.utils import setup_logging, rmse_metrics
from mlpng.utils.dataloaders import KappaDataset

setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

## Models

In [ ]:
class DeepEncoderBlock(tf.keras.layers.Layer):
    """Deep encoder block with double HEALPix Chebyshev convolution and pooling.

    Two graph convolutions per block before pooling, inspired by the double-conv
    pattern in U-Net. This gives the network more capacity to learn features at
    each resolution level before downsampling.
    """

    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        K,
        pool_p,
        max_batch_size,
        dropout_rate=0.1,
        encoder_activation="gelu",
        pool_type="AVG",
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.nside = nside

        # Select initializer based on activation
        initializer = (
            tf.keras.initializers.HeNormal()
            if encoder_activation == "relu"
            else tf.keras.initializers.GlorotNormal()
        )

        # Double convolution: fin → fout → fout, then dropout, then pool
        layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=encoder_activation,
                use_bn=True,
                # use_bias=True,
                initializer=initializer,
            ),
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=encoder_activation,
                use_bn=True,
                # use_bias=True,
                initializer=initializer,
            ),
            Dropout(dropout_rate),
            HealpyPool(pool_p, pool_type),
        ]

        self.body = HealpyGCNN(
            nside=nside,
            indices=np.arange(npix),
            layers=layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

    def call(self, x, training=False):
        return self.body(x, training=training)

In [ ]:
class TaskSpecificHeads(tf.keras.layers.Layer):
    """Shape-dependent heads with increasing capacity for local, equilateral, orthogonal."""

    SHAPE_TO_HEAD_INDEX = {
        "local": 0,
        "equilateral": 1,
        "orthogonal": 2,
    }

    # Larger heads for nside=128 deeper encoder
    HEAD_ARCHITECTURES = {
        0: [32, 32, 1],  # local (strongest signal)
        1: [128, 64, 64, 1],  # equilateral (weaker signal, needs more capacity)
        2: [32, 32, 1],  # orthogonal (weaker signal, needs more capacity)
    }

    def __init__(self, n_outputs, task_names, activation="relu", dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.n_outputs = n_outputs
        self.task_names = (
            task_names if isinstance(task_names, (list, tuple)) else [task_names]
        )
        self.activation = activation

        init = (
            tf.keras.initializers.HeNormal()
            if activation == "relu"
            else tf.keras.initializers.GlorotNormal()
        )

        def build_head(sizes, prefix, head_dropout=0.0, head_activation=None):
            layers = []
            if head_dropout > 0:
                layers.append(Dropout(head_dropout, name=f"{prefix}_dropout_0"))
            act = head_activation or activation
            for i, size in enumerate(sizes[:-1]):
                layers.append(
                    Dense(
                        size,
                        activation=act,
                        kernel_initializer=init,
                        name=f"{prefix}_dense_{i+1}",
                    )
                )
            layers.append(
                Dense(sizes[-1], kernel_initializer=init, name=f"{prefix}_output")
            )
            return layers

        self.heads = {}
        for shape_name in self.task_names:
            if shape_name in self.SHAPE_TO_HEAD_INDEX:
                head_idx = self.SHAPE_TO_HEAD_INDEX[shape_name]
                head_arch = self.HEAD_ARCHITECTURES[head_idx]
                self.heads[shape_name] = build_head(
                    head_arch, f"head_{shape_name}", dropout, activation
                )
            else:
                logger.warning(f"Unknown shape: {shape_name}, skipping head creation")

    def call(self, x, training=False):
        outputs = []
        for shape_name in self.task_names:
            if shape_name in self.heads:
                out = x
                for layer in self.heads[shape_name]:
                    out = (
                        layer(out, training=training)
                        if isinstance(layer, Dropout)
                        else layer(out)
                    )
                outputs.append(out)
        return tf.keras.layers.Concatenate()(outputs) if outputs else x


def build_deep_task_model(
    nside,
    npix,
    npol,
    n_outputs,
    task_names,
    max_batch_size=64,
    pool_p=1,
    K_schedule=None,
    channels=None,
    encoder_activation="gelu",
    head_activation="gelu",
    pool_type="AVG",
    dropout_rate=0.1,
    head_dropout=0.05,
):
    """Build deep encoder with double-conv blocks, progressive K, and task-specific heads.

    Uses pool_p=1 (halving nside each level) for maximum depth.
    For nside=128 this gives 7 encoder blocks (14 graph convolutions total).

    Architecture per block:
        HealpyChebyshev(K, fin→fout) → HealpyChebyshev(K, fout→fout) → Dropout → HealpyPool

    Args:
        nside: HEALPix nside parameter
        npix: Number of pixels (12 * nside^2)
        npol: Number of polarization channels
        n_outputs: Number of output values
        task_names: List of shape names for task-specific heads
        max_batch_size: Maximum batch size for HealpyGCNN
        pool_p: Pooling parameter (1 = halve nside each level)
        K_schedule: List of K values per level. If None, uses progressive schedule.
        channels: List of channel sizes [fin, fout_0, fout_1, ...]. If None, auto-computed.
        encoder_activation: Activation function for encoder blocks
        head_activation: Activation function for task heads
        pool_type: Pooling type ("AVG" or "MAX")
        dropout_rate: Dropout rate in encoder blocks
        head_dropout: Dropout rate in task heads
    """
    nside_factor = 2**pool_p
    depth = int(math.log(nside, nside_factor))
    level_nsides = [nside // (nside_factor**i) for i in range(depth + 1)]
    level_npixels = [12 * ns**2 for ns in level_nsides]

    # Default channel schedule: gradual increase, capped at 256
    if channels is None:
        # channels = [npol] + [2 ** (i + 4) for i in range(depth)]
        channels = [npol] + [32 for i in range(depth)]

    # Default K schedule: progressive increase with depth
    if K_schedule is None:
        K_schedule = [5 for i in range(depth)]

    print(f"Deep encoder architecture (depth={depth}):")
    print(f"  Level nsides:  {level_nsides}")
    print(f"  Level npixels: {level_npixels}")
    print(f"  Channels:      {channels}")
    print(f"  K schedule:    {K_schedule}")
    print(
        f"  Flatten size:  {level_npixels[-1]} x {channels[-1]} = {level_npixels[-1] * channels[-1]}"
    )

    inputs = tf.keras.Input(shape=(npix, npol), name="unlensed_maps")
    x = inputs

    for i in range(depth):
        x = DeepEncoderBlock(
            nside=level_nsides[i],
            npix=level_npixels[i],
            fin=channels[i],
            fout=channels[i + 1],
            K=K_schedule[i],
            pool_p=pool_p,
            max_batch_size=max_batch_size,
            dropout_rate=dropout_rate,
            encoder_activation=encoder_activation,
            pool_type=pool_type,
        )(x)

    x = Flatten()(x)
    outputs = TaskSpecificHeads(
        n_outputs=n_outputs,
        task_names=task_names,
        activation=head_activation,
        dropout=head_dropout,
    )(x)

    model = tf.keras.Model(inputs, outputs, name="deep_task_encoder")
    return model

In [ ]:
def create_sigma_weighted_loss(sigma_all):
    """
    Create a custom MSE loss function weighted by inverse sigma (precision weighting).

    Each output is normalized by its corresponding sigma value, so outputs with
    smaller sigma (higher precision) have their errors weighted more heavily.
    This is equivalent to maximum likelihood estimation for Gaussian errors.

    Args:
        sigma_all: Array of sigma values, one per output shape

    Returns:
        Loss function that takes (y_true, y_pred) and returns weighted MSE
    """
    sigma_tensor = tf.constant(sigma_all, dtype=tf.float32)

    def sigma_weighted_mse(y_true, y_pred):
        # Compute squared error per output
        squared_error = tf.square(y_true - y_pred)  # Shape: (batch_size, n_outputs)

        # Weight by inverse variance (1 / sigma^2)
        # Outputs with smaller sigma get higher weight
        weights = 1.0 / (sigma_tensor**2)

        # Apply weights and compute mean
        weighted_loss = squared_error * weights
        return tf.reduce_mean(weighted_loss)

    return sigma_weighted_mse

## Setup and Training

In [ ]:
# Initialize Core with nside 64 and all shapes
core = Core(["./settings/n64.json", "--shapes", "all", "--nsims", "10000"])
sigma_all = np.array(core.get_likelihoods(True))
custom_loss = create_sigma_weighted_loss(sigma_all)

print(f"Configuration:")
print(f"  nside: {core.nside}")
print(f"  npix:  {core.npix}")
print(f"  shapes: {core.shapes}")
print(f"  n_outputs: {len(core.shapes)}")
print(f"  Sigma values: {sigma_all}")

In [ ]:
import h5py

with h5py.File(core.file, mode="r", swmr=True, locking=False) as f:
    print(np.array(f.get("marginal_likelihoods")["unlensed"]))
    fish = f.get("fisher")["unlensed"]
    for i in fish:
        print(i, 1 / np.sqrt(np.mean(fish[i])))
    # print(np.array(1/np.sqrt(f.get("fisher")['lensed']['local'])))

In [ ]:
print(f"Configuration:")
print(f"  nside: {core.nside}")
print(f"  npix:  {core.npix}")
print(f"  shapes: {core.shapes}")
print(f"  n_outputs: {len(core.shapes)}")
print(f"  Sigma {core.shapes}: {sigma_all}")

In [ ]:
DATA_FRACTION = 1.0
DUPLICATES = [10, 10, 2]  # Train, val, test
MAX_EPOCHS = 50
PATIENCE = 8
BATCH_SIZE = 256
DECAY_STEPS = core.total_sims * 0.8 * DUPLICATES[0] * DATA_FRACTION // BATCH_SIZE
DECAY_STEPS *= 3

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CACHE_FILE = f"/lustre/smuexa01/client/users/stevensonb/tf_cache/unlensed/n{core.nside}_{'_'.join(core.shapes)}_d10_10_2/"
os.makedirs(CACHE_FILE, exist_ok=True)
print(f"Timestamp: {timestamp}")
print(f"Cache directory: {CACHE_FILE}")

ds = KappaDataset.fromCore(core, x_output="unlensed", y_output="fnl")

train_ds, val_ds, test_ds = ds.split(
    train_size=0.8 * DATA_FRACTION,
    val_size=0.1 * DATA_FRACTION,
    test_size=0.1 * DATA_FRACTION,
    to_tf=True,
    batch_size=BATCH_SIZE,
    duplicates=DUPLICATES,
    gen_batch_size=8,  # Reduced for larger nside=128 maps
    cache_file=CACHE_FILE,
    shuffle=True,
    buffer_size=2048,
)

In [ ]:
# Load best parameters from optuna study
import optuna
from pathlib import Path

# Load optuna study from database
storage_file = Path("./data/tuner") / f"optuna-n{core.nside}-2.db"
# storage_url = f"sqlite:///{storage_file.absolute()}"

# study_name = f"neo-mse-n{core.nside}-{('_').join(core.shapes)}"
# print(f"Loading optuna study: {study_name}")

# study = optuna.create_study(
#     direction="minimize",
#     study_name=study_name,
#     storage=storage_url,
#     load_if_exists=True,
# )

# best_params = study.best_trial.params
# best_value = study.best_value

# print(f"\nBest trial found: {study.best_trial.number}")
# print(f"Best validation loss: {best_value:.6f}")
# print(f"\nBest Hyperparameters:")
# for key, val in best_params.items():
#     print(f"  {key}: {val}")

In [ ]:
MODEL_TYPE = "deep_task"

# Deep model parameters for nside=128
POOL_P = 2  # Halve nside each level → 7 levels deep (128→64→32→16→8→4→2→1)
ENCODER_ACTIVATION = "gelu"
HEAD_ACTIVATION = "gelu"
POOL_TYPE = "AVG"
DROPOUT_RATE = 0.1
HEAD_DROPOUT = 0.0

K_SCHEDULE = None  # Uses default [3, 3, 3, 3, 3, 3, 3]

# Channel schedule
CHANNELS = None  # Use default (auto-computed from depth)

# Learning rate and optimization parameters
USE_COSINE_DECAY = False
INITIAL_LR = 1e-4
DECAY_RATE = 0.98
WEIGHT_DECAY = 1e-7

print(f"\nDeep Model Configuration (nside={core.nside}):")
print(f"  MODEL_TYPE: {MODEL_TYPE}")
print(f"  POOL_P: {POOL_P} (nside halved each level)")
print(f"  K_SCHEDULE: {K_SCHEDULE}")
print(f"  ENCODER_ACTIVATION: {ENCODER_ACTIVATION}")
print(f"  HEAD_ACTIVATION: {HEAD_ACTIVATION}")
print(f"  POOL_TYPE: {POOL_TYPE}")
print(f"  DROPOUT_RATE: {DROPOUT_RATE}")
print(f"  HEAD_DROPOUT: {HEAD_DROPOUT}")
print(f"  USE_COSINE_DECAY: {USE_COSINE_DECAY}")
print(f"  INITIAL_LR: {INITIAL_LR:.2e}")
print(f"  DECAY_RATE: {DECAY_RATE}")
print(f"  WEIGHT_DECAY: {WEIGHT_DECAY:.2e}")

# Setup distributed training strategy for multi-GPU
strategy = tf.distribute.MirroredStrategy()
n_gpus = strategy.num_replicas_in_sync
print(f"\nUsing MirroredStrategy with {n_gpus} device(s)")

with strategy.scope():
    model = build_deep_task_model(
        nside=core.nside,
        npix=core.npix,
        npol=core.npols,
        n_outputs=len(core.shapes),
        task_names=core.shapes,
        max_batch_size=BATCH_SIZE,
        pool_p=POOL_P,
        K_schedule=K_SCHEDULE,
        channels=CHANNELS,
        encoder_activation=ENCODER_ACTIVATION,
        head_activation=HEAD_ACTIVATION,
        pool_type=POOL_TYPE,
        dropout_rate=DROPOUT_RATE,
        head_dropout=HEAD_DROPOUT,
    )

In [ ]:
model.summary(expand_nested=True, show_trainable=True)

In [ ]:
use_cosine_decay = USE_COSINE_DECAY
initial_lr = INITIAL_LR
decay_rate = DECAY_RATE
weight_decay = WEIGHT_DECAY

# # Initialize Weights & Biases logging with all config
# wandb.init(
#     project="mlpng",
#     entity="mlpng",
#     config={
#         # Data parameters
#         "nside": core.nside,
#         "npix": core.npix,
#         "npols": core.npols,
#         "shapes": core.shapes,
#         "n_outputs": len(core.shapes),
#         "nsims": core.total_sims,
#         "fnl_min": core.fnl_min,
#         "fnl_max": core.fnl_max,
#         "sigma_values": sigma_all.tolist(),
#         "data_fraction": DATA_FRACTION,
#         "batch_size": BATCH_SIZE,
#         "decay_steps": DECAY_STEPS,
#         "max_epochs": MAX_EPOCHS,
#         "patience": PATIENCE,
#         # Deep model parameters
#         "model_type": MODEL_TYPE,
#         "K_schedule": K_SCHEDULE,
#         "head_architectures": TaskSpecificHeads.HEAD_ARCHITECTURES,
#         # Optimizer parameters
#         "use_cosine_decay": use_cosine_decay,
#         "initial_lr": initial_lr,
#         "decay_rate": decay_rate,
#         "weight_decay": weight_decay,
#         "loss": "mse",
#     },
#     tags=[f"nside-{core.nside}", MODEL_TYPE, "deep", *core.shapes],
# )

# Compile model within strategy scope for distributed training
with strategy.scope():
    if use_cosine_decay:
        lr_schedule = CosineDecayRestarts(
            initial_learning_rate=initial_lr,
            first_decay_steps=DECAY_STEPS,
            t_mul=2.0,
            m_mul=decay_rate,
            alpha=0.001,
        )
    else:
        from tensorflow.keras.optimizers.schedules import ExponentialDecay

        lr_schedule = ExponentialDecay(
            initial_learning_rate=initial_lr,
            decay_steps=DECAY_STEPS,
            decay_rate=decay_rate,
            staircase=True,
        )

    optimizer = AdamW(learning_rate=lr_schedule, weight_decay=weight_decay)

    model.compile(
        optimizer=optimizer,
        loss="mse",  # custom_loss,  # "mse",
        metrics=rmse_metrics(core.shapes),
    )

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    # WandbMetricsLogger(log_freq="epoch"),
]

model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
print(f"Training complete! Best val_loss: {min(history.history['val_loss']):.6f}")

## Full Analysis

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Val", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training History")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale("log")

# Per-shape RMSE
for shape in core.shapes:
    train_key = f"rmse_{shape}"
    val_key = f"val_rmse_{shape}"
    if train_key in history.history:
        axes[1].plot(
            history.history[val_key], label=shape, marker="o", markersize=3, alpha=0.7
        )

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")
axes[1].set_title("Val RMSE per Shape")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_predictions = model.predict(test_ds, verbose=0)

# Collect ground truth from test set
test_truth = []
for _, y in test_ds:
    test_truth.append(y.numpy())
test_truth = np.concatenate(test_truth, axis=0)
print(test_truth.shape)

# Replicate truth values across all output dimensions (same fnl for each shape)
nshapes = len(core.shapes)
# test_truth = np.tile(test_truth.reshape(-1, 1), (1, nshapes))

print(f"Predictions shape: {test_predictions.shape}")
print(f"Ground truth shape: {test_truth.shape}")

In [ ]:
# Per-shape evaluation and plots
nshapes = len(core.shapes)
sigma_all = core.get_likelihoods(True)
print(f"Sigma values: {sigma_all}")

fig, axes = plt.subplots(2, nshapes, figsize=(5 * nshapes, 10))

if nshapes == 1:
    axes = axes.reshape(2, 1)

for shape_idx, shape_name in enumerate(core.shapes):
    truth = test_truth[:, shape_idx]
    pred = test_predictions[:, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]
    corr = np.corrcoef(truth, pred)[0, 1]
    rmse = np.sqrt(np.mean(error**2))

    # Scatter plot
    line = np.array([np.nanmin(truth), np.nanmax(truth)])
    axes[0, shape_idx].scatter(truth, pred, alpha=0.4, s=8)
    axes[0, shape_idx].plot(line, line, "r--", label="Perfect", linewidth=2)
    axes[0, shape_idx].plot(line, line + sigma, "g--", alpha=0.5)
    axes[0, shape_idx].plot(line, line - sigma, "g--", alpha=0.5)
    axes[0, shape_idx].set_xlabel("True fnl")
    axes[0, shape_idx].set_ylabel("Predicted fnl")
    axes[0, shape_idx].set_title(f"{shape_name}\nCorr: {corr:.3f}, RMSE: {rmse:.2f}")
    axes[0, shape_idx].grid(True, alpha=0.3)
    axes[0, shape_idx].legend(fontsize=9)

    # Error histogram
    axes[1, shape_idx].hist(error, bins=40, alpha=0.7, edgecolor="black")
    axes[1, shape_idx].axvline(
        sigma, color="g", linestyle="--", linewidth=2, label=f"+σ"
    )
    axes[1, shape_idx].axvline(
        -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ"
    )
    axes[1, shape_idx].axvline(0, color="red", linestyle="-", linewidth=1, alpha=0.5)
    axes[1, shape_idx].set_xlabel("Prediction Error")
    axes[1, shape_idx].set_ylabel("Count")
    axes[1, shape_idx].set_title(f"Error Distribution")
    axes[1, shape_idx].grid(True, alpha=0.3, axis="y")
    axes[1, shape_idx].legend(fontsize=9)

fig.suptitle("Test Set Results - All Shapes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Summary metrics table
import pandas as pd

summary = []
for shape_idx, shape_name in enumerate(core.shapes):
    truth = test_truth[:, shape_idx]
    pred = test_predictions[:, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    summary.append(
        {
            "Shape": shape_name,
            "MAE": np.mean(np.abs(error)),
            "RMSE": np.sqrt(np.mean(error**2)),
            "RMSE/Sigma": np.sqrt(np.mean(error**2)) / sigma,
            "Correlation": np.corrcoef(truth, pred)[0, 1],
        }
    )

summary_df = pd.DataFrame(summary)
print("\nTest Set Performance Summary")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

## Restricted FNL Range Analysis

In [ ]:
# Configure restricted fnl range
FNL_MIN = -100
FNL_MAX = 100

# Filter test set to restricted range - INDEPENDENTLY for each shape
# Each shape has its own mask based on its truth values
shape_masks = {}
for shape_idx, shape_name in enumerate(core.shapes):
    shape_truth = test_truth[:, shape_idx]
    shape_masks[shape_name] = (shape_truth >= FNL_MIN) & (shape_truth <= FNL_MAX)

print(f"Restricted FNL Range: [{FNL_MIN}, {FNL_MAX}]")
print(f"Sigma values: {sigma_all}")
print(f"\nSamples in range per shape:")
for shape_name, mask in shape_masks.items():
    print(
        f"  {shape_name}: {mask.sum()} / {len(test_truth)} ({100*mask.sum()/len(test_truth):.1f}%)"
    )

In [ ]:
# Summary metrics for restricted range (each shape filtered independently)
restricted_summary = []
for shape_idx, shape_name in enumerate(core.shapes):
    mask = shape_masks[shape_name]
    truth = test_truth[mask, shape_idx]
    pred = test_predictions[mask, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    restricted_summary.append(
        {
            "Shape": shape_name,
            "N": mask.sum(),
            "MAE": np.mean(np.abs(error)),
            "RMSE": np.sqrt(np.mean(error**2)),
            "RMSE/Sigma": np.sqrt(np.mean(error**2)) / sigma,
            "Correlation": np.corrcoef(truth, pred)[0, 1] if len(truth) > 1 else np.nan,
            "Sigma": sigma,
        }
    )

restricted_df = pd.DataFrame(restricted_summary)
print(
    f"\nRestricted Range [{FNL_MIN}, {FNL_MAX}] Performance Summary (per-shape filtering)"
)
print("=" * 100)
print(restricted_df.to_string(index=False))
print("=" * 100)

# Print sigma interpretation
print("\nSigma Interpretation (measurement uncertainty from likelihood analysis):")
for row in restricted_summary:
    ratio = row["RMSE/Sigma"]
    status = "below σ" if ratio < 1.0 else "above σ"
    print(
        f"  {row['Shape']:12s}: N={row['N']:4d}, σ = {row['Sigma']:.2f}, RMSE/σ = {ratio:.3f} ({status})"
    )

In [ ]:
# Visualization for restricted range (each shape filtered independently)
nshapes = len(core.shapes)

fig, axes = plt.subplots(2, nshapes, figsize=(5 * nshapes, 10))

if nshapes == 1:
    axes = axes.reshape(2, 1)

for shape_idx, shape_name in enumerate(core.shapes):
    mask = shape_masks[shape_name]
    truth = test_truth[mask, shape_idx]
    pred = test_predictions[mask, shape_idx]
    error = pred - truth

    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]
    corr = np.corrcoef(truth, pred)[0, 1] if len(truth) > 1 else np.nan
    rmse = np.sqrt(np.mean(error**2))

    # Scatter plot
    line = np.array([FNL_MIN, FNL_MAX])
    axes[0, shape_idx].scatter(truth, pred, alpha=0.5, s=12)
    axes[0, shape_idx].plot(line, line, "r--", label="Perfect", linewidth=2)
    axes[0, shape_idx].plot(
        line, line + sigma, "g--", alpha=0.5, label=f"±σ ({sigma:.1f})"
    )
    axes[0, shape_idx].plot(line, line - sigma, "g--", alpha=0.5)
    axes[0, shape_idx].set_xlabel("True fnl")
    axes[0, shape_idx].set_ylabel("Predicted fnl")
    axes[0, shape_idx].set_title(
        f"{shape_name} (N={mask.sum()})\nCorr: {corr:.3f}, RMSE: {rmse:.2f}"
    )
    axes[0, shape_idx].set_xlim(FNL_MIN - 1, FNL_MAX + 1)
    axes[0, shape_idx].set_ylim(FNL_MIN - sigma - 1, FNL_MAX + sigma + 1)
    axes[0, shape_idx].grid(True, alpha=0.3)
    axes[0, shape_idx].legend(fontsize=9)

    # Error histogram
    axes[1, shape_idx].hist(error, bins=30, alpha=0.7, edgecolor="black")
    axes[1, shape_idx].axvline(
        sigma, color="g", linestyle="--", linewidth=2, label=f"+σ"
    )
    axes[1, shape_idx].axvline(
        -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ"
    )
    axes[1, shape_idx].axvline(0, color="red", linestyle="-", linewidth=1, alpha=0.5)
    axes[1, shape_idx].set_xlabel("Prediction Error")
    axes[1, shape_idx].set_ylabel("Count")
    axes[1, shape_idx].set_title(f"Error Distribution (σ={sigma:.1f})")
    axes[1, shape_idx].grid(True, alpha=0.3, axis="y")
    axes[1, shape_idx].legend(fontsize=9)

fig.suptitle(
    f"Restricted Range [{FNL_MIN}, {FNL_MAX}] - Test Set Results (per-shape filtering)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison: Full range vs Restricted range (per-shape filtering)
comparison_data = []
for shape_idx, shape_name in enumerate(core.shapes):
    # Full range metrics
    full_truth = test_truth[:, shape_idx]
    full_pred = test_predictions[:, shape_idx]
    full_error = full_pred - full_truth
    sigma = sigma_all[shape_idx] if len(sigma_all) > shape_idx else sigma_all[0]

    # Restricted range metrics (per-shape filtering)
    mask = shape_masks[shape_name]
    rest_truth = test_truth[mask, shape_idx]
    rest_pred = test_predictions[mask, shape_idx]
    rest_error = rest_pred - rest_truth

    comparison_data.append(
        {
            "Shape": shape_name,
            "Full N": len(full_truth),
            "Full RMSE": np.sqrt(np.mean(full_error**2)),
            "Full RMSE/σ": np.sqrt(np.mean(full_error**2)) / sigma,
            f"[{FNL_MIN},{FNL_MAX}] N": mask.sum(),
            f"[{FNL_MIN},{FNL_MAX}] RMSE": np.sqrt(np.mean(rest_error**2)),
            f"[{FNL_MIN},{FNL_MAX}] RMSE/σ": np.sqrt(np.mean(rest_error**2)) / sigma,
            "σ": sigma,
        }
    )

comparison_df = pd.DataFrame(comparison_data)
print("\nComparison: Full Range vs Restricted Range (per-shape filtering)")
print("=" * 120)
print(comparison_df.to_string(index=False))
print("=" * 120)

In [ ]:
# Log final metrics to W&B and finish run
final_metrics = {
    "best_val_loss": min(history.history["val_loss"]),
    "final_epoch": len(history.history["loss"]),
}

# Log full range test metrics
for row in summary:
    shape = row["Shape"]
    final_metrics[f"test/{shape}_mae"] = row["MAE"]
    final_metrics[f"test/{shape}_rmse"] = row["RMSE"]
    final_metrics[f"test/{shape}_rmse_sigma"] = row["RMSE/Sigma"]
    final_metrics[f"test/{shape}_correlation"] = row["Correlation"]

# Log restricted range test metrics
for row in restricted_summary:
    shape = row["Shape"]
    final_metrics[f"test_restricted/{shape}_mae"] = row["MAE"]
    final_metrics[f"test_restricted/{shape}_rmse"] = row["RMSE"]
    final_metrics[f"test_restricted/{shape}_rmse_sigma"] = row["RMSE/Sigma"]
    final_metrics[f"test_restricted/{shape}_correlation"] = row["Correlation"]

final_metrics["restricted_fnl_min"] = FNL_MIN
final_metrics["restricted_fnl_max"] = FNL_MAX

wandb.log(final_metrics)
wandb.finish()